# Unified Legal Reasoning Framework on Colab T4
Nguyen et al. (2025) — rule-based + abductive + case-based reasoning with LLMs.

**Runtime → Change runtime type → T4 GPU**, then run cells top to bottom. Each section is independent and resumable.

## 0. Setup

In [ ]:
import os, sys, subprocess
REPO_URL = ""   # e.g. https://github.com/<you>/unified-legal-reasoning ; leave empty to upload the zip instead
if REPO_URL:
    !git clone {REPO_URL} unified-legal-reasoning
else:
    from google.colab import files
    up = files.upload()          # upload unified-legal-reasoning.zip
    !unzip -q -o unified-legal-reasoning.zip
%cd unified-legal-reasoning
!pip -q install -e . transformers accelerate bitsandbytes peft pyyaml
import torch; print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

In [ ]:
# Optional: persist outputs to Drive so a quota cut / disconnect does not lose results
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/ulr_outputs'; os.makedirs(OUT, exist_ok=True)
!rm -rf outputs && ln -s {OUT} outputs

## 1. Dry run with the offline mock backend (no GPU needed, checks the whole flow)

In [ ]:
!python scripts/run_civil.py --backend mock
!python scripts/run_common.py --backend mock

## 2. Civil-law simulation with a real LLM (A.1 → A.2 → judge → A.3)
`--judge paper` reproduces the paper's judge (accepts the defendant's hypothesis). Use `--judge interactive` to play the judge yourself.

In [ ]:
!python scripts/run_civil.py --judge paper --out outputs/civil.json

## 3. Common-law simulation (B.1 → B.2 → judge → B.3)
Add `--rag` to retrieve precedents/statutes from `data/corpus` with BM25.

In [ ]:
!python scripts/run_common.py --judge paper --out outputs/common.json

In [ ]:
print(open('outputs/civil.md').read()[:3000])

## 4. §4.2 fine-tuned logit scorer (Qwen2.5-0.5B + LoRA)
Trains `Logit = Agg(O(Encode(F,T,H,O)))` on the small synthetic set in `data/logit` (extend it for real use).

In [ ]:
!python scripts/train_logit.py

In [ ]:
!python scripts/run_civil.py --judge paper --scorer outputs/logit_scorer --out outputs/civil_scored.json

## 5. §4.4 evaluation (LLM-as-a-Judge) — resumable
Re-run this cell after a disconnect; finished responses/scores are skipped. Frees each model before loading the next.

In [ ]:
!python scripts/run_eval.py --stage generate

In [ ]:
!python scripts/run_eval.py --stage judge

In [ ]:
!python scripts/run_eval.py --stage report
from IPython.display import Image, display
display(Image('outputs/eval/fig4_ranking.png')); display(Image('outputs/eval/fig5_heatmap.png'))

## 6. ALQAC-XAI — thử nghiệm khả năng GIẢI THÍCH trên dữ liệu dạng ALQAC
Outcome (`verdict_label`) được đưa sẵn cho mô hình; ta đánh giá chất lượng giải thích, **không** đánh giá độ chính xác dự đoán. Xem `docs/ALQAC_XAI.md`.

In [ ]:
!python scripts/inspect_alqac.py            # schema, kích thước, điều luật resolve được, tỉ lệ rò rỉ

In [ ]:
!python scripts/run_alqac_xai.py --backend mock --limit 1   # dry run không cần GPU

In [ ]:
!python scripts/run_alqac_xai.py --theory-source oracle   # mô hình thật (Qwen2.5-3B); resumable

In [ ]:
!python scripts/run_alqac_xai.py --theory-source oracle --decontaminate   # ablation: bỏ câu case_fact trùng court_reasoning

In [ ]:
print(open('outputs/xai/oracle/report.md').read())